# Ask CIP plan writer: training notebook

**What this does.** Teaches an existing open model (Qwen2.5 / Qwen3, Apache-2.0, already trained by Alibaba: nothing is built from scratch) to be Ask CIP's *plan writer*: it reads an analyst's
question and writes the plan Ask's engine computes exactly: which survey question is counted, among whom, split by what,
both / either / not, funnels, top lists. The model never writes a number; the engine does.

**Upload (cell 1 asks for them):** `pairs.jsonl.gz`, `ask_hard.jsonl`, `ask_heldout.jsonl`, `ask_heldout2.jsonl`
from `~/Downloads/Collab/ask-plan-writer` (made by `scripts/ask_plan_pairs.py`: every plan checked on the real data).

**Runtime:** *Runtime → Change runtime type → L4 GPU* (or A100). T4 is too small for the rewording model.

| Step | What | L4 time (estimate) |
|---|---|---|
| 1 | Setup, Drive, uploads | 2 min |
| 2 | Install | 2–4 min |
| 3 | Reword each of the 22,800 seeds into 3 analyst questions (Qwen2.5-7B-Instruct) | about 1 h (about 20 min with vLLM on an A100) |
| 4 | Build training examples | 1 min |
| 5–7 | Fine-tune and score four ready-made open models (Qwen2.5 0.5B and 1.5B, Qwen3 1.7B and 0.6B) on held-out questions never seen | 3–4 h |
| 7b | Score two ready-made models **with no training** (Qwen3-4B-Instruct-2507, Qwen3-1.7B), given 5 worked examples | 10 min |
| 8 | Export the chosen model (GGUF q8_0) and download | 10 min |

**Nothing is lost on a disconnect.** Everything is written to Google Drive (`MyDrive/ask-plan-writer/RUN`) as it goes;
run all cells again and each step continues where it stopped. The results and the model also download to your
computer's **Downloads** folder at the end. No paid AI service is used.

**Already running the first version?** Upload this one into the same run (`RUN = "v1"`) and Run all: rewording and any model already trained are skipped; only the new models and the no-training comparison run.

In [ ]:
# 1. Setup: GPU, a folder on Google Drive that survives a disconnect, and the four input files
import os, sys, json, gzip, random, re, time, subprocess
RUN = "v1"   # change to start a fresh run in a new folder
try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    WORK = f"/content/drive/MyDrive/ask-plan-writer/{RUN}"
except Exception as e:   # not on Colab, or Drive refused: a local folder (lost on disconnect)
    files = None
    print("Drive not mounted:", e)
    WORK = os.environ.get("ASK_PLAN_WORK", f"/content/ask-plan-writer/{RUN}")
os.makedirs(WORK, exist_ok=True)
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:   # no GPU driver (the repo's dry run)
    gpu = ""
print("GPU:", gpu or "none")
GPU_GB = max([int(m) for m in re.findall(r"(\d+) MiB", gpu)] or [0]) / 1024
INPUTS = ["pairs.jsonl.gz", "ask_hard.jsonl", "ask_heldout.jsonl", "ask_heldout2.jsonl"]
missing = [f for f in INPUTS if not os.path.exists(f"{WORK}/{f}")]
if missing and files:
    print("Choose these files from Downloads/Collab/ask-plan-writer:", missing)
    for name, data in files.upload().items():
        open(f"{WORK}/{name}", "wb").write(data)
missing = [f for f in INPUTS if not os.path.exists(f"{WORK}/{f}")]
assert not missing, f"still missing: {missing}"
print("work folder:", WORK)

In [ ]:
# 2. Install: transformers + peft for training (the tested path: transformers 5.24, peft 0.22, checked 23 Nov 2026)
DRY_RUN = os.environ.get("ASK_PLAN_DRY_RUN") == "1"   # the repo's own check runs steps 4 and 6 without a GPU
# vLLM rewords ~3x faster but can replace Colab's PyTorch under the training libraries; off by default.
# Set True on an A100 to save about 40 minutes of step 3.
USE_VLLM = False
if not DRY_RUN:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "transformers>=5.0", "peft>=0.20", "accelerate>=1.0", "sentencepiece"], check=True)
    if USE_VLLM:
        USE_VLLM = subprocess.run([sys.executable, "-m", "pip", "-q", "install", "vllm"]).returncode == 0
    print("vLLM:", USE_VLLM)

In [ ]:
# 3. Reword each seed into 3 questions an analyst would type (saved as it goes; resumes after a disconnect)
REWORD_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # Apache-2.0; Qwen2.5-3B is research-licence only, so not used
PER_SEED = 3
pairs = [json.loads(l) for l in gzip.open(f"{WORK}/pairs.jsonl.gz", "rt", encoding="utf-8")]
OUT = f"{WORK}/reworded.jsonl"
done = {json.loads(l)["id"] for l in open(OUT)} if os.path.exists(OUT) else set()
todo = [p for p in pairs if p["id"] not in done]
print(f"{len(pairs)} seeds, {len(done)} reworded already, {len(todo)} to go")

TIME_HINTS = ["", "", "Mention the week as 'latest survey'.", "Mention the week as 'this week'.",
              "Mention a week such as 'on {date}'.", "Ask for 'each of the last four weeks'.", "Ask about 'the last 4 surveys pooled'."]

def reword_messages(p, rng):
    hint = rng.choice(TIME_HINTS).format(date=p["week"])
    return [{"role": "system", "content": "You write questions the way market-research analysts type them into a survey tool."},
            {"role": "user", "content":
             f"Rewrite this into {PER_SEED} different questions an analyst might type. Keep exactly the same meaning: the same answers, "
             "groups, 'among', 'both', 'either', 'not', funnel steps, top lists and comparisons. Use everyday business wording instead of "
             "repeating the quoted survey names (for example 'GenAI shoppers' for people who said 'Have used GenAI to shop'). "
             "Make one short and terse, one natural, one detailed. " + hint +
             " Do not add facts or numbers. Reply with a JSON array of strings only.\n\nSeed: " + p["seed"]}]

def parse_list(text):
    m = re.search(r"\[.*\]", text, re.S)
    try:
        items = json.loads(m[0]) if m else []
    except Exception:
        items = re.findall(r'"([^"]{8,300})"', m[0]) if m else []
    return [s.strip() for s in items if isinstance(s, str) and 4 <= len(s.split()) <= 60][:PER_SEED]

if todo and not DRY_RUN:
    assert GPU_GB >= 20, "Rewording needs an L4 or A100 (Runtime → Change runtime type)"
    rng = random.Random(7)
    try:
        assert USE_VLLM
        from vllm import LLM, SamplingParams
        llm = LLM(REWORD_MODEL, dtype="bfloat16", max_model_len=2048, gpu_memory_utilization=0.9)
        params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=260)
        for start in range(0, len(todo), 2000):
            chunk = todo[start:start + 2000]
            outs = llm.chat([reword_messages(p, rng) for p in chunk], params, use_tqdm=False)
            with open(OUT, "a") as f:
                for p, o in zip(chunk, outs):
                    f.write(json.dumps({"id": p["id"], "rewrites": parse_list(o.outputs[0].text)}) + "\n")
            print(f"{start + len(chunk)} / {len(todo)}", time.strftime("%H:%M:%S"), flush=True)
    except Exception as e:   # vLLM missing or failing: transformers, batched (slower, same result)
        print("vLLM not used:", repr(e)[:200])
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(REWORD_MODEL, padding_side="left")
        model = AutoModelForCausalLM.from_pretrained(REWORD_MODEL, dtype=torch.bfloat16, device_map="cuda")
        done = {json.loads(l)["id"] for l in open(OUT)} if os.path.exists(OUT) else set()
        todo = [p for p in pairs if p["id"] not in done]
        for start in range(0, len(todo), 48):
            chunk = todo[start:start + 48]
            texts = [tok.apply_chat_template(reword_messages(p, rng), tokenize=False, add_generation_prompt=True) for p in chunk]
            batch = tok(texts, return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                gen = model.generate(**batch, max_new_tokens=260, do_sample=True, temperature=0.8, top_p=0.95)
            outs = tok.batch_decode(gen[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
            with open(OUT, "a") as f:
                for p, o in zip(chunk, outs):
                    f.write(json.dumps({"id": p["id"], "rewrites": parse_list(o)}) + "\n")
            if start % 960 == 0:
                print(f"{start + len(chunk)} / {len(todo)}", time.strftime("%H:%M:%S"), flush=True)
        del model
    import gc, torch; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4. Training examples: the question and its candidates in, the compact plan out (the same prompt Ask uses at answer time)
SYSTEM = ("You turn an analyst's question about Coresight's consumer surveys into a plan for the survey engine. "
          "Use only the numbered candidates. Reply with the plan as JSON only.")

def prompt(question, candidates):
    lines = [f"{c['n']}) {c['code']} | {c['name']} | {'; '.join(c['answers'])}" for c in candidates]
    return f"Question: {question}\nCandidates:\n" + "\n".join(lines)

def target(plan):
    return json.dumps(plan, ensure_ascii=False, separators=(",", ":"))

rewrites = {}
if os.path.exists(f"{WORK}/reworded.jsonl"):
    for l in open(f"{WORK}/reworded.jsonl"):
        r = json.loads(l); rewrites[r["id"]] = r["rewrites"]
real = {name: [json.loads(l) for l in open(f"{WORK}/{name}.jsonl")] for name in ("ask_hard", "ask_heldout", "ask_heldout2")}
rng = random.Random(11)
val_ids = {p["id"] for p in rng.sample(pairs, max(1, len(pairs) // 50))}   # 2% of seeds held back, all their wordings
train, val = [], []
for p in pairs:
    texts = rewrites.get(p["id"], []) + ([p["seed"]] if rng.random() < 0.3 or not rewrites.get(p["id"]) else [])
    for t in texts:
        (val if p["id"] in val_ids else train).append({"question": t, "candidates": p["candidates"], "plan": p["plan"], "kind": p["kind"]})
for case in real["ask_hard"] + real["ask_heldout"]:   # real analyst wording, seen four times
    train += [{"question": case["question"], "candidates": case["candidates"], "plan": case["plan"], "kind": "real"}] * 4
rng.shuffle(train)
test = [{"question": c["question"], "candidates": c["candidates"], "plan": c["plan"], "kind": "heldout2", "id": c["id"]} for c in real["ask_heldout2"]]
print(f"train {len(train):,}  val {len(val):,}  held-out-2 test {len(test)}  (seeds reworded: {len(rewrites):,} of {len(pairs):,})")
print(prompt(train[0]["question"], train[0]["candidates"]), "\n→", target(train[0]["plan"]))

In [ ]:
# 5. Train one plan writer (LoRA on Qwen2.5 / Qwen3); checkpoints on Drive, so a disconnect resumes.
#    Qwen3's thinking is off (enable_thinking=False): the plan comes straight, as Ask needs it fast
def train_model(base, epochs, batch, max_examples=None):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
    from peft import LoraConfig, get_peft_model
    name = base.split("/")[-1]
    out = f"{WORK}/{name}"
    if os.path.exists(f"{out}/final/adapter_config.json"):
        print(name, "already trained"); return out
    tok = AutoTokenizer.from_pretrained(base)
    tok.pad_token = tok.pad_token or tok.eos_token
    end = tok.convert_tokens_to_ids("<|im_end|>")
    def encode(ex):
        head = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt(ex["question"], ex["candidates"])}],
                                       tokenize=False, add_generation_prompt=True, enable_thinking=False)
        a, b = tok(head, add_special_tokens=False)["input_ids"], tok(target(ex["plan"]), add_special_tokens=False)["input_ids"] + [end]
        return {"input_ids": a + b, "labels": [-100] * len(a) + b} if len(a) + len(b) <= 1024 else None
    rows = [r for r in map(encode, train[:max_examples]) if r]
    class Rows(torch.utils.data.Dataset):
        def __len__(self): return len(rows)
        def __getitem__(self, i): return rows[i]
    def collate(batch_rows):
        n = max(len(r["input_ids"]) for r in batch_rows)
        ids = torch.full((len(batch_rows), n), tok.pad_token_id); labels = torch.full((len(batch_rows), n), -100); mask = torch.zeros((len(batch_rows), n), dtype=torch.long)
        for i, r in enumerate(batch_rows):
            k = len(r["input_ids"]); ids[i, :k] = torch.tensor(r["input_ids"]); labels[i, :k] = torch.tensor(r["labels"]); mask[i, :k] = 1
        return {"input_ids": ids, "labels": labels, "attention_mask": mask}
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.is_bf16_supported()
    fp16 = cuda and not bf16
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16 if bf16 else torch.float16 if fp16 else torch.float32)
    model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",
                                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    accumulate = max(1, 32 // batch)
    steps = max(1, len(rows) * epochs // (batch * accumulate))
    # transformers 5 has no warmup_ratio or group_by_length (checked 23 Nov 2026): warmup as 3% of the steps
    args = TrainingArguments(output_dir=out, num_train_epochs=epochs, per_device_train_batch_size=batch, gradient_accumulation_steps=accumulate,
                             learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=max(1, steps * 3 // 100), logging_steps=50, save_steps=500,
                             save_total_limit=2, bf16=bf16, fp16=fp16, report_to=[], remove_unused_columns=False, dataloader_num_workers=2 if cuda else 0)
    trainer = Trainer(model=model, args=args, train_dataset=Rows(), data_collator=collate)
    resume = any(d.startswith("checkpoint-") for d in os.listdir(out)) if os.path.exists(out) else False
    trainer.train(resume_from_checkpoint=resume or None)
    model.save_pretrained(f"{out}/final"); tok.save_pretrained(f"{out}/final")
    del trainer, model; import gc; gc.collect()
    if cuda:
        torch.cuda.empty_cache()
    return out

In [ ]:
# 6. Score a plan writer: valid JSON, the exact plan, the right structure (operation and which question in which role)
def canonical(node):
    if isinstance(node, dict):
        return {k: (sorted(v) if k == "a" else canonical(v)) for k, v in sorted(node.items()) if k != "name"}
    if isinstance(node, list):
        return [canonical(v) for v in node]
    return node

def shape(node):
    """The plan without answer labels: the operation and which candidate plays which role."""
    if isinstance(node, dict):
        return {k: shape(v) for k, v in sorted(node.items()) if k not in ("a", "name")}
    if isinstance(node, list):
        return [shape(v) for v in node]
    return node

def score(examples, outputs):
    out = {"n": len(examples), "json": 0, "exact": 0, "shape": 0}
    for ex, text in zip(examples, outputs):
        try:
            got = json.loads(text)
        except Exception:
            continue
        out["json"] += 1
        out["exact"] += canonical(got) == canonical(ex["plan"])
        out["shape"] += shape(got) == shape(ex["plan"])
    return {k: (v if k == "n" else round(100 * v / max(1, out["n"]), 1)) for k, v in out.items()}

def generate(out_dir, base, examples, batch=32):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(f"{out_dir}/final", padding_side="left")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16 if device == "cuda" else torch.float32).to(device)
    model = PeftModel.from_pretrained(model, f"{out_dir}/final").merge_and_unload().eval()
    texts = []
    for start in range(0, len(examples), batch):
        chunk = examples[start:start + batch]
        heads = [tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt(e["question"], e["candidates"])}],
                                         tokenize=False, add_generation_prompt=True, enable_thinking=False) for e in chunk]
        enc = tok(heads, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=200, do_sample=False)
        texts += [t.strip() for t in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return texts

if DRY_RUN:   # the repo's check: the scorer agrees with itself on the gold plans
    assert score(test, [target(e["plan"]) for e in test])["exact"] == 100.0
    print("dry run: scoring works", score(test, [target(e["plan"]) for e in test]))

In [ ]:
# 7. Train and score both sizes; the held-out-2 questions were never used in training or tuning
# (base, epochs, batch). Qwen3-1.7B and 0.6B (Apache-2.0) added 23 Nov 2026: on the function-calling leaderboard's
# single-call test (BFCL Live) Qwen3-1.7B scores 74.6 against 55-70 for the 1-2B specialists (xLAM-2, Hammer), whose
# licences forbid commercial use anyway
MODELS = [("Qwen/Qwen2.5-0.5B-Instruct", 2, 16), ("Qwen/Qwen2.5-1.5B-Instruct", 1, 8),
          ("Qwen/Qwen3-1.7B", 1, 8), ("Qwen/Qwen3-0.6B", 2, 16)]
results = json.load(open(f"{WORK}/results.json")) if os.path.exists(f"{WORK}/results.json") else {}
if not DRY_RUN:
    val_sample = random.Random(3).sample(val, min(600, len(val)))
    for base, epochs, batch in MODELS:
        name = base.split("/")[-1]
        if name in results:
            print(name, "scored already:", results[name]); continue
        out = train_model(base, epochs, batch)
        held = generate(out, base, test)
        results[name] = {"heldout2": score(test, held), "val": score(val_sample, generate(out, base, val_sample)),
                         "heldout2_outputs": [{"id": e["id"], "question": e["question"], "want": target(e["plan"]), "got": g} for e, g in zip(test, held)]}
        json.dump(results, open(f"{WORK}/results.json", "w"), indent=1)
        print(name, {k: v for k, v in results[name].items() if k != "heldout2_outputs"})
for name, r in results.items():
    val_part = f"  |  val exact {r['val']['exact']}%  shape {r['val']['shape']}%" if r.get("val") else ""
    print(f"{name:40} held-out-2 exact {r['heldout2']['exact']}%  shape {r['heldout2']['shape']}%{val_part}")

In [ ]:
# 7b. Ready-made models, no training: can an existing open model write the plan from 5 worked examples alone?
#     The same 22 held-out-2 questions; if one matches the fine-tuned models, Ask can use it as it is
ZERO_SHOT = ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen3-1.7B"]   # Apache-2.0; the best plain-JSON models at 4B and 2B (BFCL Live 76.4, 74.6)

def first_json(text):
    """The first {...} object in a model's reply (a ready-made model may add words around it)."""
    start = text.find("{")
    depth = 0
    for i, ch in enumerate(text[start:], start) if start >= 0 else []:
        depth += ch == "{"
        depth -= ch == "}"
        if depth == 0:
            return text[start:i + 1]
    return text

def worked_examples(k=5):
    """One real analyst question per operation, from the training side (never held-out-2)."""
    picked, seen = [], set()
    for ex in train:
        kind = (ex["plan"].get("op"), "among" in ex["plan"], "by" in ex["plan"], any(x in json.dumps(ex["plan"]) for x in ('"both"', '"either"', '"not"')))
        if ex["kind"] == "real" and kind not in seen:
            seen.add(kind); picked.append(ex)
        if len(picked) == k:
            break
    return "\n\n".join(prompt(e["question"], e["candidates"]) + "\nPlan: " + target(e["plan"]) for e in picked)

if not DRY_RUN:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    shots = worked_examples()
    for base in ZERO_SHOT:
        key = f"zero-shot {base.split('/')[-1]}"
        if key in results:
            print(key, "scored already"); continue
        tok = AutoTokenizer.from_pretrained(base, padding_side="left")
        model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.bfloat16, device_map="cuda").eval()
        texts = []
        for start in range(0, len(test), 8):
            chunk = test[start:start + 8]
            heads = [tok.apply_chat_template([{"role": "system", "content": SYSTEM},
                                              {"role": "user", "content": "Examples:\n\n" + shots + "\n\nNow this one:\n" + prompt(e["question"], e["candidates"]) + "\nPlan:"}],
                                             tokenize=False, add_generation_prompt=True, enable_thinking=False) for e in chunk]
            enc = tok(heads, return_tensors="pt", padding=True).to("cuda")
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=220, do_sample=False)
            texts += [first_json(t.strip()) for t in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
        results[key] = {"heldout2": score(test, texts),
                        "heldout2_outputs": [{"id": e["id"], "question": e["question"], "want": target(e["plan"]), "got": g} for e, g in zip(test, texts)]}
        json.dump(results, open(f"{WORK}/results.json", "w"), indent=1)
        print(key, results[key]["heldout2"])
        del model; torch.cuda.empty_cache()


In [ ]:
# 8. Export the better model (held-out-2 shape, then exact) as GGUF q8_0 for the server and download the results
if not DRY_RUN and results:
    tuned = [n for n in results if not n.startswith("zero-shot")]
    best = max(tuned, key=lambda n: (results[n]["heldout2"]["shape"], results[n]["heldout2"]["exact"], -("1.5B" in n or "1.7B" in n)))
    base = next(b for b, _, _ in MODELS if b.endswith(best))
    print("exporting", best)
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    merged = f"{WORK}/{best}/merged"
    if not os.path.exists(f"{merged}/config.json"):
        model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16)
        PeftModel.from_pretrained(model, f"{WORK}/{best}/final").merge_and_unload().save_pretrained(merged)
        AutoTokenizer.from_pretrained(f"{WORK}/{best}/final").save_pretrained(merged)
    LLAMA = os.environ.get("ASK_PLAN_LLAMA", "/content/llama.cpp")   # llama.cpp's converter turns the model into one GGUF file
    if not os.path.exists(LLAMA):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", f"{LLAMA}/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
    gguf = f"{WORK}/ask-plan-writer-{best}-q8_0.gguf"
    if not os.path.exists(gguf):
        subprocess.run([sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", merged, "--outtype", "q8_0", "--outfile", gguf], check=True)
    json.dump({"best": best, "prompt_system": SYSTEM, "results": {k: {x: y for x, y in v.items() if x != "heldout2_outputs"} for k, v in results.items()}},
              open(f"{WORK}/summary.json", "w"), indent=1)
    print("saved on Drive:", WORK)
    if files:
        for path in (f"{WORK}/summary.json", f"{WORK}/results.json", gguf):
            files.download(path)   # to your computer's Downloads folder